# Setup Your Digest

This notebook helps you fork the repo for a new research area.

It walks through:
1. Choosing arXiv keywords with sample queries and hit counts.
2. Testing the triage LLM on 5 sample papers.
3. Generating a topic config file in `configs/topics/`.


## Imports

This uses the repo's existing modules rather than notebook-only helper code.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

from eegfm_digest.arxiv import category_match, dedupe_latest, fetch_query
from eegfm_digest.config import DEFAULT_OPENROUTER_MODEL
from eegfm_digest.keywords import ARXIV_CATEGORIES, default_topic_payload
from eegfm_digest.llm import LLMCallConfig, build_llm_call, load_api_key
from eegfm_digest.triage import load_schema, triage_paper


## 1. Choose arXiv keywords

Start with a few broad candidate queries, then compare recent hit counts after category filtering.

The counts below are not the full arXiv total. They are counts from the most recent `max_results` items returned by the API, which is usually enough to compare query quality while iterating.


In [ ]:
sample_queries = {
    "current_query_a": 'all:(eeg OR electroencephalograph* OR brainwave*) AND all:("foundation model" OR pretrain OR pretrained OR "self-supervised" OR "self supervised")',
    "current_query_b": 'all:(eeg OR electroencephalograph* OR brainwave*) AND all:("representation learning" OR masked OR transfer OR generaliz*)',
    "broader_ssl": 'all:(eeg OR electroencephalograph* OR brainwave*) AND all:("self-supervised" OR pretrain OR transfer learning)',
    "task_specific_example": 'all:(sleep EEG OR seizure EEG) AND all:(pretrain OR foundation model OR representation learning)',
}


def recent_hit_count(query: str, max_results: int = 50) -> int:
    rows = fetch_query(query, max_results=max_results, rate_limit_seconds=0.0)
    rows = [row for row in rows if category_match(row["categories"], allowed_categories=ARXIV_CATEGORIES)]
    return len(dedupe_latest(rows))


for name, query in sample_queries.items():
    print(f"\n{name}")
    print(query)
    print("recent filtered hits:", recent_hit_count(query, max_results=50))


After you find promising retrieval queries, pick one query to inspect qualitatively. The next cell pulls 5 recent examples after category filtering.


In [ ]:
selected_query_name = "current_query_a"
selected_query = sample_queries[selected_query_name]

candidate_rows = fetch_query(selected_query, max_results=20, rate_limit_seconds=0.0)
candidate_rows = [row for row in candidate_rows if category_match(row["categories"], allowed_categories=ARXIV_CATEGORIES)]
candidate_rows = dedupe_latest(candidate_rows)[:5]

for idx, paper in enumerate(candidate_rows, start=1):
    print(f"\n[{idx}] {paper['arxiv_id_base']} | {paper['title']}")
    print("categories:", ", ".join(paper["categories"]))
    print("abstract:", paper["summary"][:400], "...")


## 2. Test the triage LLM on 5 sample papers

This uses the same triage schema and prompt as the main pipeline. The model is the repo default, and the API key is loaded from the same environment variable path as normal runs.


In [ ]:
triage_schema = load_schema(Path("schemas/triage.json"))
triage_prompt = Path("prompts/triage.md").read_text(encoding="utf-8")
repair_prompt = Path("prompts/repair_json.md").read_text(encoding="utf-8")

llm = build_llm_call(
    LLMCallConfig(
        provider="openrouter",
        api_key=load_api_key(),
        model=DEFAULT_OPENROUTER_MODEL,
        temperature=0.2,
        max_output_tokens=1024,
        base_url="https://openrouter.ai/api/v1",
    )
)

triage_results = []
try:
    for paper in candidate_rows:
        decision = triage_paper(paper, llm, triage_prompt, repair_prompt, triage_schema)
        triage_results.append(
            {
                "arxiv_id_base": paper["arxiv_id_base"],
                "title": paper["title"],
                "decision": decision["decision"],
                "confidence": decision["confidence"],
                "reasons": decision["reasons"],
            }
        )
finally:
    llm.close()

triage_results


## 3. Generate a topic config file

Fill in the fields below and run the cell to write `configs/topics/<slug>.json`.


In [ ]:
topic_slug = "my_topic"
topic_title = "My Research Digest"
topic_description = "Monthly arXiv digest for my research area"

topic_queries = [
    sample_queries["current_query_a"],
    sample_queries["current_query_b"],
]

topic_categories = sorted(ARXIV_CATEGORIES)

topic_config = default_topic_payload()
topic_config.update(
    {
        "slug": topic_slug,
        "title": topic_title,
        "description": topic_description,
        "categories": topic_categories,
        "queries": topic_queries,
    }
)

out_path = Path("configs/topics") / f"{topic_slug}.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(topic_config, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

print(f"Wrote topic config to {out_path}")
print(json.dumps(topic_config, ensure_ascii=False, indent=2))


## Next step

Run the pipeline against your new topic config:

```bash
export GEMINI_API_KEY="..."
export TOPIC_CONFIG_PATH="configs/topics/my_topic.json"
python -m eegfm_digest.run --month 2026-03
```
